In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('data.csv', sep='\t')
df.head()

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,4778106097,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,URN:catalog:CLO:EBIRD:OBS1830446111,Animalia,Chordata,Aves,Accipitriformes,Accipitridae,Milvus,Milvus migrans,...,NaN,NaN,CC_BY_4_0,NaN,obsr3708998,NaN,NaN,2025-11-07T06:03:55.287Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_CONCE...
1,3638002563,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,URN:catalog:CLO:EBIRD:OBS1192443748,Animalia,Chordata,Aves,Passeriformes,Pycnonotidae,Pycnonotus,Pycnonotus cafer,...,NaN,NaN,CC_BY_4_0,NaN,obsr1287009,NaN,NaN,2025-11-07T06:00:15.464Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
2,5666076977,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,URN:catalog:CLO:EBIRD:OBS744062151,Animalia,Chordata,Aves,Passeriformes,Aegithinidae,Aegithina,Aegithina tiphia,...,NaN,NaN,CC_BY_4_0,NaN,obsr450692,NaN,NaN,2025-11-07T06:22:16.240Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_CONCE...
3,5501141114,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,URN:catalog:CLO:EBIRD:OBS2366578480,Animalia,Chordata,Aves,NaN,NaN,Pachyglossa,NaN,...,NaN,NaN,CC_BY_4_0,NaN,obsr3930558,NaN,NaN,2025-11-07T06:28:18.159Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_CONCE...
4,4258016175,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,URN:catalog:CLO:EBIRD:OBS1496957767,Animalia,Chordata,Aves,Coraciiformes,Alcedinidae,Pelargopsis,Pelargopsis capensis,...,NaN,NaN,CC_BY_4_0,NaN,obsr1091489,NaN,NaN,2025-11-07T06:01:56.886Z,NaN,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...


In [5]:
df.shape

(2382899, 50)

In [6]:
print(df['locality'].value_counts().head(100).to_string())

locality
Yala National Park (General)                                                                         85548
Bundala NP (General)                                                                                 72880
Matara--Kirala Kele Sanctuary                                                                        50189
Uda Walawe NP                                                                                        38657
Sinharaja Forest Reserve                                                                             37100
Attidiya                                                                                             36033
Home garden                                                                                          34372
Thalagama Lake (and surrounding wetlands)                                                            27606
wekunagoda                                                                                           24532
Udugama.Western Province-Hom

In [7]:
import pandas as pd

# remove missing coordinates
df = df.dropna(subset=["decimalLatitude", "decimalLongitude"])

# create grid cells
df["lat_grid"] = df["decimalLatitude"].round(2)
df["lon_grid"] = df["decimalLongitude"].round(2)

# count points per grid
grid_counts = df.groupby(["lat_grid", "lon_grid"]).size().reset_index(name="count")

print(grid_counts.head())

   lat_grid  lon_grid  count
0      5.89     80.52     28
1      5.89     80.53      8
2      5.90     80.44     39
3      5.90     80.49     12
4      5.90     80.53      5


In [8]:
import folium
import matplotlib.colors as mcolors
import numpy as np

m = folium.Map(location=[7.87, 80.77], zoom_start=7)

# Create a logarithmic color scale to handle skewed data
min_count = grid_counts["count"].min()
max_count = grid_counts["count"].max()
norm = mcolors.LogNorm(vmin=min_count, vmax=max_count)
cmap_list = ['green', 'yellow', 'orange', 'red', 'darkred']

# Create colormap
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('density', cmap_list)

for _, row in grid_counts.iterrows():
    # Get color based on log-scaled count
    color_value = norm(row["count"])
    rgba = cmap(color_value)
    color_hex = mcolors.rgb2hex(rgba)
    
    # Use log scale for marker radius too
    radius = np.log1p(row["count"]) / 2  # log scale prevents huge markers
    
    folium.CircleMarker(
        location=[row["lat_grid"], row["lon_grid"]],
        radius=max(2, radius),  # minimum radius of 2
        color=color_hex,
        fill=True,
        fill_color=color_hex,
        fill_opacity=0.6,
        popup=f"Count: {row['count']}"
    ).add_to(m)

m.save("bird_density_map.html")